# El asistente de viaje, paso a paso

Una palanca por celda. La pregunta no cambia nunca:

> Revisa mis correos de la última semana y dime cuándo es mi próximo vuelo.

En la primera celda el modelo contesta que no puede. En la última contesta el
vuelo de LATAM del jueves 24 de septiembre, 06:15, BOG → MDE, reserva ABC123.

Aquí se usa `await`, no `asyncio.run`: el notebook ya tiene su event loop.

In [ ]:
import entorno  # carga ANTHROPIC_API_KEY de .env
import sys
from pathlib import Path

from claude_agent_sdk import (
    AgentDefinition, ClaudeAgentOptions, HookMatcher,
    create_sdk_mcp_server, query, tool,
)

from mostrar import linea

AQUI = Path.cwd()
PREGUNTA = "Revisa mis correos de la última semana y dime cuándo es mi próximo vuelo."
SYSTEM_PROMPT = """Eres un asistente personal. Respondes en español, corto y con datos.
Usa tus tools y skills; no adivines fechas ni reservas."""

# Lo que no cambia en todo el recorrido: el modelo, los topes y el aislamiento
# de tu configuración personal de Claude Code.
BASE = dict(
    model="claude-sonnet-5",
    system_prompt=SYSTEM_PROMPT,
    setting_sources=[],
    strict_mcp_config=True,
    env={"CLAUDE_CODE_DISABLE_AUTO_MEMORY": "1"},
    max_turns=12,
    max_budget_usd=0.25,
    stderr=lambda _: None,
)


async def correr(pregunta=PREGUNTA, **palancas):
    """Corre el agente con BASE más las palancas de esta celda."""
    opciones = ClaudeAgentOptions(**{**BASE, **palancas})
    async for m in query(prompt=pregunta, options=opciones):
        linea(m)

## 01 · `query()`

Sin tools. Un prompt entra, una respuesta sale, el proceso termina. El modelo
dice que no puede ver tu correo: ese "no puedo" es el punto de partida.

In [ ]:
await correr(tools=[])

## 02 · el stream

`query()` no devuelve un texto: devuelve un flujo de mensajes. `linea()` los
imprime con su etiqueta: ARRANQUE, EL LLM PIDE, LA TOOL RESPONDE, EL LLM DICE,
RECIBO. En el recibo, `num_turns` cuenta vueltas del loop, no turnos de chat.

In [ ]:
@tool("hoy", "La fecha de hoy. Pídela siempre antes de razonar sobre fechas.", {})
async def hoy(args):
    return {"content": [{"type": "text", "text": "sábado 19 de septiembre de 2026"}]}


RELOJ = create_sdk_mcp_server(name="reloj", version="1.0.0", tools=[hoy])

await correr(tools=[], mcp_servers={"reloj": RELOJ}, allowed_tools=["mcp__reloj__hoy"])

## 03 · una tool propia

`@tool` describe la función para el LLM, `create_sdk_mcp_server` la empaqueta y
la clave de `mcp_servers` da el nombre completo: `mcp__reloj__hoy`.

Dos listas distintas: `tools` dice qué existe, `allowed_tools` qué se aprueba
sin preguntar. `allowed_tools` no restringe nada.

In [ ]:
RELOJ_RUIDOSO = create_sdk_mcp_server(name="reloj", version="1.0.0", tools=[hoy])

# Quita "mcp__reloj__hoy" de allowed_tools y la corrida se queda esperando permiso.
await correr(tools=[], mcp_servers={"reloj": RELOJ_RUIDOSO}, allowed_tools=["mcp__reloj__hoy"])

## 04 · MCP

`gmail_mcp.py` es un servidor MCP falso con cuatro correos. Corre en otro
proceso, por stdio. Para el agente sus tools son iguales a las tuyas.

`strict_mcp_config=True` deja fuera los conectores de tu cuenta claude.ai (si no,
el agente se va a tu Gmail de verdad) y también el `.mcp.json` de la carpeta: por
eso los servidores se declaran aquí, inline.

In [ ]:
GMAIL = {"command": sys.executable, "args": [str(AQUI / "gmail_mcp.py")]}

await correr(
    tools=[],
    mcp_servers={"reloj": RELOJ, "gmail": GMAIL},
    allowed_tools=["mcp__reloj__hoy", "mcp__gmail__buscar", "mcp__gmail__leer"],
)

## 05 · un skill

El procedimiento vive en `workspace/.claude/skills/buscar-vuelo/SKILL.md`, en
markdown, y lo puede escribir alguien que no programa. `cwd` dice dónde vive el
agente y `setting_sources=["project"]` le deja leer ese `.claude/`.

El LLM solo tiene en contexto el nombre y la descripción hasta que decide usarlo.
Compara el orden de las tools con el de la celda anterior.

In [ ]:
await correr(
    cwd=str(AQUI / "workspace"),
    setting_sources=["project"],
    skills=["buscar-vuelo"],
    tools=["Skill"],
    mcp_servers={"reloj": RELOJ, "gmail": GMAIL},
    allowed_tools=["Skill", "mcp__reloj__hoy", "mcp__gmail__buscar", "mcp__gmail__leer"],
)

## 06 · permisos y hooks

`disallowed_tools` con el nombre pelado saca la tool del contexto.
`permission_mode="dontAsk"` niega lo que no esté aprobado, en vez de esperar a un
humano que en un script no existe. Y el hook `PreToolUse` es el único que ve los
**argumentos**: aquí bloquea la lectura de un correo marcado como privado.

El motivo del `deny` vuelve al LLM como resultado de la tool. No se cae: cambia
de ruta y lo dice en la respuesta.

In [ ]:
async def portero(entrada, tool_use_id, contexto):
    if entrada["tool_input"].get("id") not in {"c2", "c4"}:
        return {}
    return {"hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "deny",
        "permissionDecisionReason": "Ese correo es privado. Usa el asunto y la fecha de la lista.",
    }}


await correr(
    cwd=str(AQUI / "workspace"),
    setting_sources=["project"],
    skills=["buscar-vuelo"],
    tools=["Skill", "Read", "Bash", "Write", "Edit"],
    disallowed_tools=["Bash", "Write", "Edit"],
    permission_mode="dontAsk",
    hooks={"PreToolUse": [HookMatcher(matcher="mcp__gmail__leer", hooks=[portero])]},
    mcp_servers={"reloj": RELOJ, "gmail": GMAIL},
    allowed_tools=["Skill", "mcp__reloj__hoy", "mcp__gmail__buscar", "mcp__gmail__leer"],
)

## 07 · un subagente

`lector-de-correos` tiene su propio contexto y su propio modelo. Busca, lee y
devuelve una sola línea por vuelo. El padre nunca ve los cuatro correos completos.

Lo único que viaja del padre al hijo es el prompt de la llamada. En el stream la
tool se llama `Agent`; en la lista del ARRANQUE aparece como `Task`. Los mensajes
del hijo traen `parent_tool_use_id` y el impresor los marca `[subagente]`.

In [ ]:
LECTOR = AgentDefinition(
    description="Lee la bandeja de correo y devuelve los vuelos que encuentre.",
    prompt="Haz UNA sola búsqueda con el rango que te pidan y lee los correos que "
           "puedan ser vuelos. Devuelve una línea por vuelo: fecha, hora, ruta y "
           "código de reserva. No opines ni expliques el proceso.",
    tools=["mcp__gmail__buscar", "mcp__gmail__leer"],
    model="haiku",
    maxTurns=8,        # los campos de varias palabras van en camelCase
    background=False,
)

await correr(
    agents={"lector-de-correos": LECTOR},
    tools=["Agent"],
    forward_subagent_text=True,
    mcp_servers={"reloj": RELOJ, "gmail": GMAIL},
    allowed_tools=["Agent", "mcp__reloj__hoy", "mcp__gmail__buscar", "mcp__gmail__leer"],
)

## 08 · todo junto

Tool propia, MCP, skill, hook y los dos topes que evitan que un bucle te cueste
dinero. La respuesta correcta: jueves 24 de septiembre de 2026, 06:15,
BOG → MDE, LATAM, reserva ABC123.

Cambia la pregunta y vuelve a correr.

In [ ]:
async def solo_vuelos(entrada, tool_use_id, contexto):
    if entrada["tool_input"].get("id") != "c4":
        return {}
    return {"hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "deny",
        "permissionDecisionReason": "El correo c4 es privado. Responde solo sobre vuelos.",
    }}


await correr(
    PREGUNTA,
    cwd=str(AQUI / "workspace"),
    setting_sources=["project"],
    skills=["buscar-vuelo"],
    tools=["Skill"],
    hooks={"PreToolUse": [HookMatcher(matcher="mcp__gmail__leer", hooks=[solo_vuelos])]},
    mcp_servers={"reloj": RELOJ, "gmail": GMAIL},
    allowed_tools=["Skill", "mcp__reloj__hoy", "mcp__gmail__buscar", "mcp__gmail__leer"],
)